In [8]:
import os, re
import numpy as np
import pandas as pd
from pathlib import Path
import scanpy as sc
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt


from sklearn.decomposition import TruncatedSVD
from scipy.sparse import issparse


In [2]:
IN_H5AD = "MAIT_NKT_gdT_paperlabels_exact.h5ad"  
OUTDIR  = "feature_qc"                           

CURRENT_FEATURES_TXT = "new_run_allTF/var_names.txt"  
TF_LIST_TXT          = ""   

N_NEIGH   = 40
MIN_DIST  = 0.45

LINEAGE_COL = "paper_lineage"
SUBTYPE_COL = "paper_subtype_strict"
STAGE_COL   = "paper_stage"
METRIC    = "cosine"



os.makedirs(OUTDIR, exist_ok=True)
plt.rcParams['figure.dpi'] = 120


TCR_RE = re.compile(r'^(TRAV|TRAJ|TRBV|TRBJ|TRGV|TRGJ|TRDV|TRDJ|TR[A-Z]\d+)', re.IGNORECASE)

def drop_tcr_mask(varnames):
    return np.array([False if TCR_RE.match(g) else True for g in varnames])

def read_list_maybe(path):
    if not path or not Path(path).exists():
        return []
    return pd.read_csv(path, header=None)[0].astype(str).tolist()


In [3]:
ad = sc.read_h5ad(IN_H5AD)

ad_norm = ad.copy()
if "counts" in ad_norm.layers:
    ad_norm.X = ad_norm.layers["counts"].copy()
sc.pp.normalize_total(ad_norm, target_sum=1e4)
sc.pp.log1p(ad_norm)


F_BASE = read_list_maybe(CURRENT_FEATURES_TXT)  
TF_LIST = read_list_maybe(TF_LIST_TXT)           

len(ad_norm), ad_norm.n_vars, len(F_BASE), len(TF_LIST)


(8253, 25970, 2705, 0)

In [4]:
vn = pd.Index(ad_norm.var_names)
vn_lower_map = {g.lower(): g for g in vn}

def map_markers_to_data(markers):
    mapped = []
    missing = []
    for g in markers:
        hit = vn_lower_map.get(g.lower(), None)
        if hit is None:
            g_mouse = g[:1].upper() + g[1:].lower()
            hit = vn_lower_map.get(g_mouse.lower(), None)
        if hit is None:
            missing.append(g)
        else:
            mapped.append(hit)
    return sorted(pd.unique(mapped)), sorted(pd.unique(missing))


core_markers = ["TBX21","RORC","IFNG","IL17A","IL17F","CCR6","CXCR3","ZBTB16","ICOS","CD44","KLF2","S1PR1","CCR7","SELL","KLRG1","ITGAE","NKG7","GZMB"]

mapped_core, missing_core = map_markers_to_data(core_markers)
print("Found in data:", mapped_core)

markers_by_lineage = {
    "NKT":  ["Zbtb16","Cd44","Itga4","Cxcr3","Icos","Tbx21","Ifng","Rorc","Il17a"],
    "MAIT": ["Klf2","S1pr1","Ccr7","Sell","Cxcr6","Icos","Cd44","Tbx21","Ifng","Rorc","Il17a"],
    "gdT":  ["Tbx21","Ifng","Nkg7","Gzmb","Cxcr3","Klrg1","Itgae","Rorc","Il17a","Il17f","Ccr6"],
}

for lin, lst in markers_by_lineage.items():
    mapped, miss = map_markers_to_data(lst)
    markers_by_lineage[lin] = mapped


MARKERS = sorted({g for lst in markers_by_lineage.values() for g in lst})
len(MARKERS), MARKERS[:10]


Found in data: ['Ccr6', 'Ccr7', 'Cd44', 'Cxcr3', 'Gzmb', 'Icos', 'Ifng', 'Il17a', 'Il17f', 'Itgae', 'Klf2', 'Klrg1', 'Nkg7', 'Rorc', 'S1pr1', 'Sell', 'Tbx21', 'Zbtb16']


(20,
 ['Ccr6',
  'Ccr7',
  'Cd44',
  'Cxcr3',
  'Cxcr6',
  'Gzmb',
  'Icos',
  'Ifng',
  'Il17a',
  'Il17f'])

In [5]:
# HVG5000
ad_raw_for_hvg = ad.copy()
if "counts" in ad.layers:
    ad_raw_for_hvg.X = ad.layers["counts"].copy()
sc.pp.normalize_total(ad_raw_for_hvg, target_sum=1e4)
sc.pp.log1p(ad_raw_for_hvg)
sc.pp.highly_variable_genes(ad_raw_for_hvg, flavor="seurat_v3", n_top_genes=5000, inplace=True)
HVG5000 = ad_raw_for_hvg.var_names[ad_raw_for_hvg.var["highly_variable"]].tolist()

len(HVG5000), HVG5000[:5]


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/preprocessing/_highly_variable_genes.py:62: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(


(5000, ['Gm37988', '4732440D04Rik', 'Gm7449', 'Gm7512', 'Gm37409'])

In [6]:
def make_feature_sets(f_base, hvg5000, marker_set, addN=(100,200,300)):
    fsets = {"F_base": sorted(set(f_base))}
    for n in addN:
        fsets[f"F_+H{n}"] = sorted(set(f_base) | set(hvg5000[:n]))
    fsets["F_+markers"] = sorted(set(f_base) | set(marker_set))
    fsets["F_+H200+markers"] = sorted(set(fsets["F_+H200"]) | set(marker_set))
    return fsets

FSETS = make_feature_sets(F_BASE, HVG5000, MARKERS)

def coverage_row(name, genes, universe):
    arr = np.isin(genes, universe)
    return dict(name=name, total=len(genes), in_universe=int(arr.sum()), pct=float(arr.mean()))

cov_rows = []
if F_BASE:
    cov_rows.append(coverage_row("F_base ∩ HVG5000", F_BASE, HVG5000))
    cov_rows.append(coverage_row("markers ∩ F_base", MARKERS, F_BASE))
for k,v in FSETS.items():
    cov_rows.append(coverage_row(f"{k} ∩ HVG5000", v, HVG5000))
    cov_rows.append(coverage_row(f"markers ∩ {k}", MARKERS, v))

coverage_df = pd.DataFrame(cov_rows)
coverage_df


,name,total,in_universe,pct
0,F_base ∩ HVG5000,2705,1131,0.418115
1,markers ∩ F_base,20,18,0.900000
2,F_base ∩ HVG5000,2705,1131,0.418115
3,markers ∩ F_base,20,18,0.900000
4,F_+H100 ∩ HVG5000,2781,1207,0.434017
5,markers ∩ F_+H100,20,18,0.900000
6,F_+H200 ∩ HVG5000,2868,1294,0.451185
7,markers ∩ F_+H200,20,18,0.900000
8,F_+H300 ∩ HVG5000,2952,1378,0.466802
9,markers ∩ F_+H300,20,18,0.900000


In [7]:
coverage_df.to_csv(f"{OUTDIR}/feature_coverage.csv", index=False)


In [9]:
os.environ["NUMBA_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]   = "1"
os.environ["MKL_NUM_THREADS"]   = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"


In [10]:

def safe_neighbors(ad, n_neighbors, metric, use_rep):

    try:
        sc.pp.neighbors(ad, n_neighbors=n_neighbors, metric=metric, use_rep=use_rep, method="gauss")
    except TypeError:
        sc.pp.neighbors(ad, n_neighbors=n_neighbors, metric=metric, use_rep=use_rep)

def safe_umap(ad, min_dist=0.45, random_state=0):
    try:
        sc.tl.umap(ad, min_dist=min_dist, random_state=random_state, init_pos="spectral")
    except TypeError:
        sc.tl.umap(ad, min_dist=min_dist, random_state=random_state)

def run_umap_eval_safe(adata, genes, label_col, fig_prefix, pca_n=5):
    genes = [g for g in genes if g in adata.var_names]
    if len(genes) < 50:
        return {"ok": False, "msg": f"Too few genes ({len(genes)})", "silhouette": np.nan, "knn_acc": np.nan, "n_genes": len(genes)}
    ad = adata[:, genes].copy()

    if issparse(ad.X):
        ad.X = ad.X.tocsr().astype("float32", copy=False)
    else:
        ad.X = ad.X.astype("float32", copy=False)
    svd = TruncatedSVD(n_components=pca_n, random_state=0)
    ad.obsm["X_pca"] = svd.fit_transform(ad.X)

    safe_neighbors(ad, n_neighbors=N_NEIGH, metric=METRIC, use_rep="X_pca")
    safe_umap(ad, min_dist=MIN_DIST, random_state=0)

    emb = ad.obsm["X_umap"]
    labels = ad.obs[label_col].astype(str)

    try:
        sil = silhouette_score(emb, labels)
    except Exception:
        sil = np.nan

    idx = np.arange(ad.n_obs); np.random.shuffle(idx)
    split = int(0.7*len(idx))
    tr, te = idx[:split], idx[split:]
    knn = KNeighborsClassifier(n_neighbors=15)
    knn.fit(emb[tr], labels.iloc[tr])
    acc = float((knn.predict(emb[te]) == labels.iloc[te]).mean())

    sc.pl.umap(ad, color=[label_col], frameon=False, show=False, size=8)
    plt.savefig(Path(OUTDIR)/f"UMAP_SHARED_{fig_prefix}_{label_col}.png", dpi=300, bbox_inches="tight")
    plt.close()

    return {"ok": True, "silhouette": float(sil), "knn_acc": acc, "n_genes": len(genes)}


In [11]:
shared_rows = []
for name, genes in FSETS.items():
    res = run_umap_eval_safe(
        ad_norm,
        genes,
        SUBTYPE_COL if SUBTYPE_COL in ad_norm.obs else LINEAGE_COL,
        fig_prefix=name,
        pca_n=5
    )
    res["fset"] = name; res["level"] = "shared"
    shared_rows.append(res)

shared_df = pd.DataFrame(shared_rows)
shared_df.to_csv(f"{OUTDIR}/umap_eval_shared.csv", index=False)
shared_df.sort_values(["ok","silhouette","knn_acc"], ascending=[False,False,False])


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


,ok,silhouette,knn_acc,n_genes,fset,level
0,True,0.314422,0.940630,2705,F_base,shared
4,True,0.314183,0.935784,2707,F_+markers,shared
1,True,0.310882,0.942649,2781,F_+H100,shared
5,True,0.310882,0.944669,2870,F_+H200+markers,shared
2,True,0.309769,0.934572,2868,F_+H200,shared
3,True,0.309728,0.935380,2952,F_+H300,shared


In [11]:
ad_norm

AnnData object with n_obs × n_vars = 8253 × 25970
    obs: 'orig.ident', 'nCount_originalexp', 'nFeature_originalexp', 'nGene', 'nUMI', 'use', 'final_celltype', 'major_type', 'paper_subtype', 'n_counts', 'n_genes', 'author_cluster', 'author_lineage', 'paper_cluster', 'paper_lineage'
    var: 'gene_symbols', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'gene_symbol'
    uns: 'log1p', 'paper_subtype_colors', 'umap_provenance'
    obsm: 'X_PCA', 'X_TSNE', 'X_umap'
    layers: 'counts'

In [12]:
ad_norm.obs["paper_lineage"]

F3-AAACCTGAGAGGTTAT     NKT
F3-AAACCTGCAATCGAAA     NKT
F3-AAACGGGAGAGTCGGT     NKT
F3-AAAGATGTCCGCAAGC     NKT
F3-AAAGCAAGTGGCGAAT     NKT
                       ... 
F5-GTTCTCGTCAAGAAGT    MAIT
F5-TAGTGGTAGTCTCAAC     γδT
F5-TCCCGATGTAGCCTAT     NKT
F5-TGACTAGCAGCATACT     γδT
F5-TGAGGGAGTAGTAGTA     γδT
Name: paper_lineage, Length: 8253, dtype: category
Categories (3, object): ['NKT', 'MAIT', 'γδT']

In [6]:
SUBTYPE_COL = "paper_subtype_strict" 
LINEAGE_COL = "paper_lineage"


In [13]:
if LINEAGE_COL not in ad_norm.obs:
    def _lin_from_cluster(c):
        c=str(c)
        if c.startswith("N"): return "NKT"
        if c.startswith("M"): return "MAIT"
        if c.startswith(("G","γ")): return "γδT"
        return "NA"
    ad_norm.obs[LINEAGE_COL] = ad_norm.obs["paper_cluster"].astype(str).map(_lin_from_cluster)

if SUBTYPE_COL not in ad_norm.obs:
    if "paper_cluster" not in ad_norm.obs:
        raise KeyError("Нет paper_subtype_strict и paper_cluster — не из чего построить подтипы.")
    map_nkt   = {"N1":"NKTp","N2":"NKT1","N3":"NKT2","N4":"NKT2","N5":"NKT2","N6":"NKT2","N7":"NKT17"}
    map_mait  = {"M1":"MAIT0","M2":"MAIT2","M3":"MAIT1/17i","M4":"MAIT1i","M5":"MAIT1","M6":"MAIT17i","M7":"MAIT17i","M8":"MAIT17"}
    map_gdt   = {"G1":"Tγδp","G2":"Tγδ1/17i","G3":"Tγδ1/17i","G4":"Tγδ17i","G5":"Tγδ17i","G6-1":"Tγδ17","G6-2":"Tγδ17","G7-1":"Tγδ1i","G7-2":"Tγδ1"}
    def _subtype(cl): return map_nkt.get(cl, map_mait.get(cl, map_gdt.get(cl, "NA")))
    ad_norm.obs[SUBTYPE_COL] = ad_norm.obs["paper_cluster"].astype(str).map(_subtype)

if "paper_stage" not in ad_norm.obs and "paper_cluster" in ad_norm.obs:
    map_mait_stage = {"M1":"S1","M2":"S2","M3":"S2","M4":"S2","M5":"S3","M6":"S3","M7":"S3","M8":"S3"}
    ad_norm.obs["paper_stage"] = ad_norm.obs["paper_cluster"].astype(str).map(map_mait_stage)

ad_norm.obs[SUBTYPE_COL].value_counts().head(20)


NKT2         2567
Tγδ17i       1013
Tγδ1/17i      633
MAIT1/17i     562
MAIT0         532
MAIT17        464
Tγδ1i         419
MAIT17i       344
Tγδp          310
NKT17         283
Tγδ17         230
NKTp          225
NKT1          210
MAIT1i        154
MAIT1         148
MAIT2          83
Tγδ1           76
Name: paper_subtype_strict, dtype: int64

In [14]:

def run_umap_eval_safe(adata, genes, label_col, fig_prefix, pca_n=5,
                       n_neighbors=40, metric="cosine", min_dist=0.45, outdir="feature_qc"):
    genes = [g for g in genes if g in adata.var_names]
    if len(genes) < 50:
        return {"ok": False, "msg": f"Too few genes ({len(genes)})", "silhouette": np.nan, "knn_acc": np.nan, "n_genes": len(genes)}
    ad = adata[:, genes].copy()
    if issparse(ad.X): ad.X = ad.X.tocsr().astype("float32", copy=False)
    else:              ad.X = ad.X.astype("float32", copy=False)

    svd = TruncatedSVD(n_components=pca_n, random_state=0)
    ad.obsm["X_pca"] = svd.fit_transform(ad.X)
    safe_neighbors(ad, n_neighbors=n_neighbors, metric=metric, use_rep="X_pca")
    safe_umap(ad, min_dist=min_dist, random_state=0)

    emb = ad.obsm["X_umap"]
    labels = ad.obs[label_col].astype(str)


    try:
        sil = silhouette_score(emb, labels)
    except Exception:
        sil = np.nan

    idx = np.arange(ad.n_obs); np.random.shuffle(idx)
    tr, te = idx[:int(0.7*len(idx))], idx[int(0.7*len(idx)):]
    knn = KNeighborsClassifier(n_neighbors=15)
    knn.fit(emb[tr], labels.iloc[tr])
    acc = float((knn.predict(emb[te]) == labels.iloc[te]).mean())

    sc.pl.umap(ad, color=[label_col], frameon=False, show=False, size=8)
    Path(outdir).mkdir(exist_ok=True, parents=True)
    plt.savefig(Path(outdir)/f"UMAP_{fig_prefix}_{label_col}.png", dpi=300, bbox_inches="tight")
    plt.close()

    return {"ok": True, "silhouette": float(sil), "knn_acc": acc, "n_genes": len(genes)}


In [15]:
N_NEIGH, METRIC, MIN_DIST = 40, "cosine", 0.45
OUTDIR = "feature_qc"

lin_rows = []
for lin in ["NKT","MAIT","γδT"]:    
    m_lin = ad_norm.obs[LINEAGE_COL].astype(str).eq(lin).values
    if m_lin.sum() == 0: 
        continue

    sub_raw  = ad[m_lin].copy()
    sub_norm = ad_norm[m_lin].copy()

    keep = drop_tcr_mask(np.array(sub_raw.var_names))
    sub_raw  = sub_raw[:, keep]
    sub_norm = sub_norm[:, keep]

    sc.pp.highly_variable_genes(sub_raw, flavor="seurat_v3", n_top_genes=5000, inplace=True)
    lin_hvg = set(sub_raw.var_names[sub_raw.var["highly_variable"]].tolist())

    for name, genes in FSETS.items():
        genes_lineage = sorted(set(genes) & lin_hvg)
        if len(genes_lineage) < 50:
            lin_rows.append({"lineage": lin, "fset": name, "ok": False, "n_genes": len(genes_lineage),
                             "silhouette": np.nan, "knn_acc": np.nan})
            continue

        res = run_umap_eval_safe(
            sub_norm, genes_lineage, SUBTYPE_COL,
            fig_prefix=f"LINEAGE_{lin}_{name}_25PC",
            pca_n=25, n_neighbors=N_NEIGH, metric=METRIC, min_dist=MIN_DIST, outdir=OUTDIR
        )
        res.update({"lineage": lin, "fset": name, "level": "lineage"})
        lin_rows.append(res)

lin_df = pd.DataFrame(lin_rows)
lin_df.to_csv(f"{OUTDIR}/umap_eval_lineage.tsv", sep="\t", index=False)
lin_df.sort_values(["lineage","ok","silhouette","knn_acc"], ascending=[True,False,False,False])


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/preprocessing/_highly_variable_genes.py:62: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(
/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/anndata/compat/_overloaded_dict.py:106: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  self.data[key] = value
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


...

/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


,ok,silhouette,knn_acc,n_genes,lineage,fset,level
11,True,0.234055,0.796215,1004,MAIT,F_+H200+markers,lineage
7,True,0.233918,0.825328,957,MAIT,F_+H100,lineage
6,True,0.232405,0.823872,912,MAIT,F_base,lineage
8,True,0.231960,0.809316,1003,MAIT,F_+H200,lineage
10,True,0.231533,0.819505,913,MAIT,F_+markers,lineage
9,True,0.231268,0.828239,1051,MAIT,F_+H300,lineage
1,True,0.245052,0.959432,1363,NKT,F_+H100,lineage
3,True,0.240467,0.964503,1450,NKT,F_+H300,lineage
5,True,0.238950,0.959432,1406,NKT,F_+H200+markers,lineage
4,True,0.232146,0.964503,1333,NKT,F_+markers,lineage


In [16]:
df = pd.read_csv(f"{OUTDIR}/umap_eval_lineage.tsv", sep="\t")
df = df[df["ok"]==True].copy()

def topk(df_lin, k=3):
    return (df_lin
            .sort_values(["silhouette","knn_acc","n_genes"], ascending=[False, False, True])
            .head(k)[["fset","n_genes","silhouette","knn_acc"]])

print("NKT:\n", topk(df[df["lineage"]=="NKT"]))
print("\nMAIT:\n", topk(df[df["lineage"]=="MAIT"]))
print("\nγδT:\n", topk(df[df["lineage"]=="γδT"]))


NKT:
               fset  n_genes  silhouette   knn_acc
1          F_+H100     1363    0.245052  0.959432
3          F_+H300     1450    0.240467  0.964503
5  F_+H200+markers     1406    0.238950  0.959432

MAIT:
                fset  n_genes  silhouette   knn_acc
11  F_+H200+markers     1004    0.234055  0.796215
7           F_+H100      957    0.233918  0.825328
6            F_base      912    0.232405  0.823872

γδT:
                fset  n_genes  silhouette   knn_acc
17  F_+H200+markers     1263    0.032451  0.699379
15          F_+H300     1305    0.032415  0.709317
12           F_base     1175    0.030993  0.704348


In [17]:
mait_stage_markers = ["Klf2","S1pr1","Ccr7","Sell"]
missing_mait = [g for g in mait_stage_markers if g not in set(FSETS["F_+H100"])]
print("MAIT stage markers missing from F_+H100:", missing_mait)

MAIT stage markers missing from F_+H100: ['Ccr7']


In [18]:
missing_mait = [g for g in mait_stage_markers if g not in set(FSETS["F_+H200+markers"])]
print("MAIT stage markers missing from F_+H100:", missing_mait)

MAIT stage markers missing from F_+H100: []


In [19]:
# γδT: F_+H200 ∪ γδT_HVG300 ∪ γδ-markers) \ TCR


gdt_markers = ["Tbx21","Ifng","Nkg7","Gzmb","Cxcr3","Klrg1","Itgae","Rorc","Il17a","Il17f","Ccr6"]
gdt_markers = [g for g in gdt_markers if g in ad_norm.var_names]
m_gdt = ad_norm.obs[LINEAGE_COL].astype(str).isin(["γδT","gdT"])

sub_raw  = ad[m_gdt].copy()
sub_norm = ad_norm[m_gdt].copy()


keep = drop_tcr_mask(np.array(sub_raw.var_names))
sub_raw  = sub_raw[:, keep]
sub_norm = sub_norm[:, keep]


sc.pp.highly_variable_genes(sub_raw, flavor="seurat_v3", n_top_genes=5000, inplace=True)
gdt_hvg = sub_raw.var_names[sub_raw.var["highly_variable"]].tolist()[:300]

F_gdT_boost = sorted(set(FSETS["F_+H200"]) | set(gdt_hvg) | set(gdt_markers))
print("F_gdT_boost size:", len(F_gdT_boost))


res_gdt = run_umap_eval_safe(
    sub_norm, [g for g in F_gdT_boost if g in sub_norm.var_names], SUBTYPE_COL,
    fig_prefix="LINEAGE_gdT_BOOST_25PC", pca_n=25, n_neighbors=20,  # n_neighbors поострее
    metric="cosine", min_dist=0.2, outdir="feature_qc"
)
res_gdt


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/preprocessing/_highly_variable_genes.py:62: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(
/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/anndata/compat/_overloaded_dict.py:106: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  self.data[key] = value


F_gdT_boost size: 3028


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


{'ok': True,
 'silhouette': 0.00461264792829752,
 'knn_acc': 0.6956521739130435,
 'n_genes': 3027}

In [20]:

F_NKT = FSETS["F_+H100"]

F_MAIT_min = sorted(set(FSETS["F_+H100"]) | {"Ccr7"})
res_min = run_umap_eval_safe(ad_norm[ad_norm.obs[LINEAGE_COL].eq("MAIT")], F_MAIT_min, SUBTYPE_COL,
                             fig_prefix="LINEAGE_MAIT_Fmin_25PC", pca_n=25, n_neighbors=40,
                             metric="cosine", min_dist=0.45, outdir="feature_qc")

F_MAIT_alt = FSETS["F_+H200+markers"]
res_alt = run_umap_eval_safe(ad_norm[ad_norm.obs[LINEAGE_COL].eq("MAIT")], F_MAIT_alt, SUBTYPE_COL,
                             fig_prefix="LINEAGE_MAIT_Falt_25PC", pca_n=25, n_neighbors=40,
                             metric="cosine", min_dist=0.45, outdir="feature_qc")

print("MAIT minimal:", res_min)
print("MAIT alt(H200+markers):", res_alt)


F_MAIT = F_MAIT_min if (res_min["silhouette"] >= (res_alt["silhouette"] - 0.002)) else F_MAIT_alt
print("Chosen F_MAIT size:", len(F_MAIT))


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


MAIT minimal: {'ok': True, 'silhouette': 0.23680344223976135, 'knn_acc': 0.8107714701601164, 'n_genes': 2782}
MAIT alt(H200+markers): {'ok': True, 'silhouette': 0.18400993943214417, 'knn_acc': 0.8122270742358079, 'n_genes': 2870}
Chosen F_MAIT size: 2782


In [21]:
m_gdt = ad_norm.obs[LINEAGE_COL].astype(str).isin(["γδT","gdT"])
sub_raw  = ad[m_gdt].copy()
sub_norm = ad_norm[m_gdt].copy()
keep = drop_tcr_mask(np.array(sub_raw.var_names))
sub_raw, sub_norm = sub_raw[:, keep], sub_norm[:, keep]
sc.pp.highly_variable_genes(sub_raw, flavor="seurat_v3", n_top_genes=5000, inplace=True)
gdt_hvg600 = sub_raw.var_names[sub_raw.var["highly_variable"]].tolist()[:600]

gdt_markers = [g for g in ["Tbx21","Ifng","Nkg7","Gzmb","Cxcr3","Klrg1","Itgae","Rorc","Il17a","Il17f","Ccr6"] if g in sub_norm.var_names]
F_gdT_boost2 = sorted(set(FSETS["F_+H200"]) | set(gdt_hvg600) | set(gdt_markers))

res_gdt2 = run_umap_eval_safe(sub_norm, F_gdT_boost2, SUBTYPE_COL,
                              fig_prefix="LINEAGE_gdT_BOOST2_25PC", pca_n=25, n_neighbors=20,
                              metric="cosine", min_dist=0.2, outdir="feature_qc")
print("gdT boost2:", res_gdt2)



/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/preprocessing/_highly_variable_genes.py:62: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(
/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/anndata/compat/_overloaded_dict.py:106: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  self.data[key] = value


/root/miniconda/envs/celloracle_env/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


gdT boost2: {'ok': True, 'silhouette': 0.029078597202897072, 'knn_acc': 0.7167701863354037, 'n_genes': 3254}


In [22]:
if "paper_stage" not in ad_norm.obs:
    map_mait_stage = {"M1":"S1","M2":"S2","M3":"S2","M4":"S2","M5":"S3","M6":"S3","M7":"S3","M8":"S3"}
    ad_norm.obs["paper_stage"] = ad_norm.obs["paper_cluster"].astype(str).map(map_mait_stage)


F_MAIT = sorted(set(FSETS["F_+H100"]) | {"Ccr7"})
print("F_MAIT size:", len(F_MAIT))


F_MAIT size: 2782


In [24]:
#best choice
F_NKT  = FSETS["F_+H100"]                   
F_MAIT = sorted(set(FSETS["F_+H100"]) | {"Ccr7"})  
F_gdT  = F_gdT_boost2                 

len(F_NKT), len(F_MAIT), len(F_gdT)


(2781, 2782, 3255)